In [7]:
# ---------- Load your files, align by month, compute state-dependent CCAPM betas ----------
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import os

# File names (you gave these)
ff6_file = 'FF6.csv'
cay_file = 'cay_index.csv'
market_file = 'Market-Returns-Cleaned.csv'
cons_file = 'Consumption-Cleaned.csv'
rf_file = 'Risk-Free-Cleaned.csv'

In [8]:
# ---------- Helper: parse 'Apr-14' style month into datetime ----------
def parse_month_col(df, col='month'):
    # try common patterns; coerce errors to NaT so we can inspect
    try:
        parsed = pd.to_datetime(df[col], format='%b-%y', errors='coerce')
        if parsed.isna().any():
            # try alternative like 'Apr-2014'
            parsed = pd.to_datetime(df[col], format='%b-%Y', errors='coerce')
    except Exception as e:
        parsed = pd.to_datetime(df[col], errors='coerce')
    return parsed

# ---------- 1) Load CSVs ----------
df_ff6 = pd.read_csv(ff6_file)
df_cay = pd.read_csv(cay_file)
df_market = pd.read_csv(market_file)
df_consumption = pd.read_csv(cons_file)
df_rf = pd.read_csv(rf_file)

# Preview column names and first rows (helpful for debugging)
print("FF6 columns:", df_ff6.columns.tolist())
print("CAY columns:", df_cay.columns.tolist())
print("Market columns:", df_market.columns.tolist())
print("Consumption columns:", df_consumption.columns.tolist())
print("RF columns:", df_rf.columns.tolist())

# ---------- 2) Parse month columns (assume column named 'month') ----------
for d, name in [(df_ff6, 'FF6'), (df_cay, 'CAY'), (df_market, 'Market'), (df_consumption, 'Consumption'), (df_rf, 'RF')]:
    if 'month' not in d.columns:
        raise KeyError(f"'{name}' file does not contain a 'month' column. Please ensure each CSV has a 'month' column.")
    d['month_dt'] = parse_month_col(d, 'month')
    if d['month_dt'].isna().any():
        print(f"Warning: some 'month' values in {name} could not be parsed. First 10 original values:\n", d['month'].head(10).tolist())
    d.set_index('month_dt', inplace=True)
    # drop the original textual month if you want, but keep it for reference
    # d.drop(columns=['month'], inplace=True)

# ---------- 3) Inspect and pick portfolio columns from FF6 ----------
# Heuristic: choose numeric columns in df_ff6 that are not 'month' or textual
candidate_port_cols = [c for c in df_ff6.columns if c.lower() not in ('month','month_dt') and pd.api.types.is_numeric_dtype(df_ff6[c])]
print("Candidate portfolio columns detected in FF6:", candidate_port_cols)

# If exactly 6 numeric columns found, assume these are the 6 portfolios. Otherwise ask user to specify.
if len(candidate_port_cols) == 6:
    portfolios = candidate_port_cols
else:
    # try common names if present
    common_names = ['Small_Growth','Small_Neutral','Small_Value','Big_Growth','Big_Neutral','Big_Value',
                    'SG','SN','SV','BG','BN','BV']
    portfolios = [c for c in common_names if c in df_ff6.columns]
    if len(portfolios) != 6:
        print("Could not unambiguously detect 6 portfolio columns in FF6.csv.")
        print("Please edit the `portfolios` list below to match the six portfolio column names in FF6.csv.")
        print("Detected numeric columns:", candidate_port_cols)
        # provide a fallback: pick first 6 numeric
        portfolios = candidate_port_cols[:6]
        print("Proceeding with first 6 numeric columns as portfolios:", portfolios)

# ---------- 4) Find cay column name in df_cay ----------
possible_cay_names = [c for c in df_cay.columns if 'cay' in c.lower()]
if len(possible_cay_names) >= 1:
    cay_col = possible_cay_names[0]
else:
    # pick numeric column other than month
    numeric_cols = [c for c in df_cay.columns if pd.api.types.is_numeric_dtype(df_cay[c]) and c.lower() != 'month']
    if len(numeric_cols) >= 1:
        cay_col = numeric_cols[0]
    else:
        raise KeyError("Could not find cay column in cay_index.csv. Please ensure it contains a 'cay' series.")

print("Using CAY column:", cay_col)

# ---------- 5) Find consumption column or delta_c in df_consumption ----------
possible_delta_c = [c for c in df_consumption.columns if 'delta' in c.lower() or 'growth' in c.lower() or 'deltac' in c.lower()]
if len(possible_delta_c) >= 1:
    delta_c_col = possible_delta_c[0]
    print("Found delta_c-like column in consumption file:", delta_c_col)
    # if it is already log-diff, we'll use directly as 'delta_c'
    df_consumption['delta_c'] = df_consumption[delta_c_col]
else:
    # try to find a level consumption column and compute log diff
    possible_level = [c for c in df_consumption.columns if c.lower() not in ('month','month_dt') and pd.api.types.is_numeric_dtype(df_consumption[c])]
    if len(possible_level) >= 1:
        level_col = possible_level[0]
        print(f"No explicit delta_c found. Using level column '{level_col}' to compute delta_c = logdiff(level).")
        df_consumption['delta_c'] = np.log(df_consumption[level_col]).diff()
    else:
        raise KeyError("Could not find consumption columns. Ensure Consumption-Cleaned.csv contains a consumption series or a delta_c series.")

# ---------- 6) Find risk-free column in df_rf ----------
possible_rf = [c for c in df_rf.columns if 'rf' in c.lower() or 'r_f' in c.lower() or 'risk' in c.lower()]
if len(possible_rf) >= 1:
    rf_col = possible_rf[0]
else:
    numeric_rf = [c for c in df_rf.columns if pd.api.types.is_numeric_dtype(df_rf[c]) and c.lower() not in ('month','month_dt')]
    if len(numeric_rf) >= 1:
        rf_col = numeric_rf[0]
    else:
        raise KeyError("Could not detect risk-free column in Risk-Free-Cleaned.csv. Please ensure it contains a numeric rf series.")
print("Using RF column:", rf_col)

# ---------- 7) Merge dataframes on month index ----------
# We'll build a master df that contains portfolios, cay, delta_c, rf
# Start with FF6 portfolios (df_ff6 indexed by month_dt)
master = df_ff6[portfolios].copy()

# Merge cay
master = master.join(df_cay[[cay_col]], how='inner')
master.rename(columns={cay_col: 'cay'}, inplace=True)

# Merge consumption delta (delta_c)
if 'delta_c' not in df_consumption.columns:
    raise KeyError("delta_c not found in df_consumption after earlier steps.")
master = master.join(df_consumption[['delta_c']], how='inner')

# Merge risk-free
master = master.join(df_rf[[rf_col]], how='inner')
master.rename(columns={rf_col: 'rf'}, inplace=True)

# Optionally merge market returns too (not needed for this CCAPM plot, but helpful)
# try to detect market return column
possible_market = [c for c in df_market.columns if 'market' in c.lower() or 'nse' in c.lower() or 'index' in c.lower() or 'return' in c.lower()]
if len(possible_market) >= 1:
    market_col = possible_market[0]
    master = master.join(df_market[[market_col]], how='left')
    master.rename(columns={market_col: 'market_return'}, inplace=True)
    print("Merged market returns column:", market_col)
else:
    print("Market returns column not found automatically in Market-Returns-Cleaned.csv. Skipping merge of market returns.")

# Drop rows with missing values (you might prefer inner join behaviour)
master = master.dropna().sort_index()
print(f"Master series length after merge and dropna: {len(master)} months (from {df_ff6.shape[0]} original rows)")

# ---------- 8) Sanity checks ----------
print(master[['cay','delta_c','rf']].head(6))
print("Portfolio sample columns:", portfolios)

# NOTE: If rf appears annualized (e.g., values > 1), convert to monthly:
# Uncomment and adjust if needed:
# if master['rf'].abs().max() > 1:
#     print("Risk-free values appear >1 (likely annual). Converting to monthly approx via rf_month = rf_annual / 12.")
#     master['rf'] = master['rf'] / 12.0

# ---------- 9) Create t+1 aligned variables (we use cay_t to define state; regress R_{t+1} on delta_c_{t+1}) ----------
m = master.copy()
for p in portfolios:
    m[f'{p}_next'] = m[p].shift(-1)
m['rf_next'] = m['rf'].shift(-1)
m['delta_c_next'] = m['delta_c'].shift(-1)

# Drop last row(s) where next is NaN
m2 = m.dropna(subset=[f'{portfolios[0]}_next', 'rf_next', 'delta_c_next', 'cay']).copy()
print(f"Effective observations for CCAPM state regression: {len(m2)}")

# Excess returns at t+1
for p in portfolios:
    m2[f'{p}_excess_next'] = m2[f'{p}_next'] - m2['rf_next']

# Define good / bad state using median of cay (full-sample)
cay_median = m2['cay'].median()
m2['is_good'] = (m2['cay'] >= cay_median).astype(int)
m2['is_bad']  = (m2['cay'] <  cay_median).astype(int)
print(f"Using cay median {cay_median:.6g}: good months = {m2['is_good'].sum()}, bad months = {m2['is_bad'].sum()}")

# ---------- 10) Estimate state-dependent CCAPM betas ----------
import collections
beta_good = collections.OrderedDict()
beta_bad = collections.OrderedDict()
se_good = collections.OrderedDict()
se_bad = collections.OrderedDict()

for p in portfolios:
    y = m2[f'{p}_excess_next'].values
    X = np.column_stack([
        np.ones(len(m2)),                               # intercept
        m2['delta_c_next'].values * m2['is_good'].values,  # delta_c_{t+1} * I_good(t)
        m2['delta_c_next'].values * m2['is_bad'].values    # delta_c_{t+1} * I_bad(t)
    ])
    ols = sm.OLS(y, X, missing='drop')
    res = ols.fit(cov_type='HC1')  # robust SE
    beta_good[p] = res.params[1]
    beta_bad[p]  = res.params[2]
    se_good[p]   = res.bse[1]
    se_bad[p]    = res.bse[2]

# ---------- 11) Prepare summary DataFrame ----------
summary = pd.DataFrame({
    'portfolio': portfolios,
    'beta_good': [beta_good[p] for p in portfolios],
    'se_good': [se_good[p] for p in portfolios],
    'beta_bad': [beta_bad[p] for p in portfolios],
    'se_bad': [se_bad[p] for p in portfolios]
})

print("\nState-dependent CCAPM beta summary:")
print(summary.round(4))

# ---------- 12) Plot Good vs Bad betas ----------
fig, ax = plt.subplots(figsize=(7,7))
x = summary['beta_good'].values
y = summary['beta_bad'].values
ax.scatter(x, y, s=70)
for i, lab in enumerate(summary['portfolio']):
    ax.annotate(lab, (x[i], y[i]), textcoords="offset points", xytext=(6,4), fontsize=9)
# 45-degree line
lo = min(np.min(x), np.min(y)) - 0.05 * abs(min(np.min(x), np.min(y), 1))
hi = max(np.max(x), np.max(y)) + 0.05 * abs(max(np.max(x), np.max(y), 1))
ax.plot([lo, hi], [lo, hi], linestyle='--', color='gray')
ax.set_xlabel('Good-state consumption beta (β^G)')
ax.set_ylabel('Bad-state consumption beta (β^B)')
ax.set_title('State-dependent CCAPM Betas: Good vs Bad States')
ax.grid(True)
plt.tight_layout()

os.makedirs('figures', exist_ok=True)
fig_path = os.path.join('figures', 'state_ccapm_betas.png')
plt.savefig(fig_path, dpi=300)
print(f"Saved plot to {fig_path}")

# Save CSV summary
csv_path = 'state_ccapm_beta_summary.csv'
summary.to_csv(csv_path, index=False)
print(f"Saved summary CSV to {csv_path}")

plt.show()
# ---------- end ----------


FF6 columns: ['month', 'Portfolios BV returns (%)', 'Portfolios BN returns (%)', 'Portfolios BG returns (%)', 'Portfolios SV returns (%)', 'Portfolios SN returns (%)', 'Portfolios SG returns (%)']
CAY columns: ['month', 'cay']
Market columns: ['month', 'market_return']
Consumption columns: ['month', 'weighted_avg_consumption']
RF columns: ['month', 'Yeild']
 ['2014-04', '2014-05', '2014-06', '2014-07', '2014-08', '2014-09', '2014-10', '2014-11', '2014-12', '2015-01']
 ['2014-04', '2014-05', '2014-06', '2014-07', '2014-08', '2014-09', '2014-10', '2014-11', '2014-12', '2015-01']
Candidate portfolio columns detected in FF6: ['Portfolios BV returns (%)', 'Portfolios BN returns (%)', 'Portfolios BG returns (%)', 'Portfolios SV returns (%)', 'Portfolios SN returns (%)', 'Portfolios SG returns (%)']
Using CAY column: cay
No explicit delta_c found. Using level column 'weighted_avg_consumption' to compute delta_c = logdiff(level).
Using RF column: Yeild
Merged market returns column: market_retu

ValueError: zero-size array to reduction operation maximum which has no identity